# Week 11 - Kiểm thử, tối ưu hệ thống và độ ổn định mô hình

Notebook này dùng cho giai đoạn sau khi đã train model ở các tuần trước. Trọng tâm không phải là thêm kiến trúc mới, mà là kiểm tra xem pipeline có đáng tin cậy không:

- Dữ liệu có nhất quán không: đủ split, đúng label range, không leak giữa train/val/test.
- Checkpoint có load đúng kiến trúc không, output shape có đúng `num_classes` không.
- Kết quả đánh giá có ổn định khi chạy lặp lại không.
- Hệ thống data loading và inference có điểm nghẽn rõ ràng không.
- Kết quả thực nghiệm nên được đọc như thế nào: metric tổng quan, per-class accuracy, confusion pairs, ví dụ dự đoán sai.

Tinh thần của notebook: giúp sinh viên hiểu foundation của kiểm thử mô hình ML trước khi tối ưu sâu hơn.


## 1. Chuẩn bị môi trường

Các cell dưới đây cố tình giữ code tường minh. Với project học thuật, việc thấy rõ từng bước kiểm thử thường quan trọng hơn việc đóng gói quá sớm thành framework riêng.


In [ ]:
from __future__ import annotations

import json
import os
import random
import time
import warnings
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.transforms import InterpolationMode

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

warnings.filterwarnings("ignore", category=FutureWarning)


In [ ]:
ARTIFACTS_DIR = Path("./artifacts/datasets")
CHECKPOINT_DIR = Path("./artifacts/checkpoints")
HISTORY_DIR = Path("./artifacts/training")

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
HISTORY_DIR.mkdir(parents=True, exist_ok=True)

CLASS_TO_IDX_PATH = ARTIFACTS_DIR / "class_to_idx.json"
TRAIN_RECORDS_PATH = ARTIFACTS_DIR / "train_records.json"
VAL_RECORDS_PATH = ARTIFACTS_DIR / "val_records.json"
TEST_RECORDS_PATH = ARTIFACTS_DIR / "test_records.json"

IMAGE_SIZE = 224
RESIZE_SIZE = 256
BATCH_SIZE = 64
NUM_WORKERS = 8

# Benchmark vừa đủ để thấy xu hướng, không biến notebook kiểm thử thành một benchmark suite nặng.
BENCHMARK_WORKERS = [0, 2, 4, 8, 12]
BENCHMARK_BATCHES = 60
INFERENCE_BENCHMARK_BATCHES = 40

# Giới hạn phần stability/robustness để notebook chạy được trên laptop GPU phổ thông.
STABILITY_REPEATS = 3
STABILITY_MAX_BATCHES = 8
TTA_SAMPLES = 4
TTA_MAX_IMAGES = 128
ERROR_EXAMPLE_COUNT = 12

USE_BBOX = False
BBOX_PADDING = 0.05

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

RANDOM_SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EVAL_USE_AMP = False

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")

print(f"Device      : {DEVICE}")
print(f"CPU threads : {os.cpu_count()}")
print(f"Batch size  : {BATCH_SIZE}")
print(f"Workers     : {NUM_WORKERS}")
print(f"Eval AMP    : {EVAL_USE_AMP}")


## 2. Utilities, dataset và DataLoader

Cell này giữ lại cùng preprocessing evaluation của các tuần trước: `Resize(256) -> CenterCrop(224) -> Normalize(ImageNet)`. Khi kiểm thử mô hình, ta không dùng augmentation ngẫu nhiên cho metric chính vì cần kết quả dễ lặp lại.


In [ ]:
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_json(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {path}")
    return json.loads(path.read_text(encoding="utf-8"))


def parse_annotation(annotation_path: str) -> dict:
    root = ET.parse(annotation_path).getroot()
    size_node = root.find("size")
    object_node = root.find("object")
    bbox_node = object_node.find("bndbox")
    return {
        "width": int(size_node.findtext("width")),
        "height": int(size_node.findtext("height")),
        "bbox": {
            "xmin": int(bbox_node.findtext("xmin")),
            "ymin": int(bbox_node.findtext("ymin")),
            "xmax": int(bbox_node.findtext("xmax")),
            "ymax": int(bbox_node.findtext("ymax")),
        },
    }


def crop_with_bbox(image: Image.Image, bbox: dict, padding_ratio: float = 0.0) -> Image.Image:
    xmin, ymin, xmax, ymax = bbox["xmin"], bbox["ymin"], bbox["xmax"], bbox["ymax"]
    box_width = xmax - xmin
    box_height = ymax - ymin
    pad_x = int(box_width * padding_ratio)
    pad_y = int(box_height * padding_ratio)
    xmin = max(0, xmin - pad_x)
    ymin = max(0, ymin - pad_y)
    xmax = min(image.width, xmax + pad_x)
    ymax = min(image.height, ymax + pad_y)
    return image.crop((xmin, ymin, xmax, ymax))


class StanfordDogsDataset(Dataset):
    def __init__(
        self,
        records: list[dict],
        transform=None,
        use_bbox: bool = False,
        bbox_padding: float = 0.0,
    ) -> None:
        self.records = list(records)
        self.transform = transform
        self.use_bbox = use_bbox
        self.bbox_padding = bbox_padding

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int):
        record = self.records[index]
        with Image.open(record["image_path"]) as pil_image:
            image = pil_image.convert("RGB")
        if self.use_bbox:
            annotation = parse_annotation(record["annotation_path"])
            image = crop_with_bbox(image, annotation["bbox"], padding_ratio=self.bbox_padding)
        if self.transform is not None:
            image = self.transform(image)
        return image, record["class_id"]


def dataloader_kwargs(num_workers: int) -> dict:
    kwargs = {
        "num_workers": num_workers,
        "pin_memory": torch.cuda.is_available(),
    }
    if num_workers > 0:
        kwargs["persistent_workers"] = True
        kwargs["prefetch_factor"] = 2
    return kwargs


def make_loader(
    records: list[dict],
    transform,
    batch_size: int = BATCH_SIZE,
    shuffle: bool = False,
    num_workers: int = NUM_WORKERS,
) -> DataLoader:
    dataset = StanfordDogsDataset(
        records,
        transform=transform,
        use_bbox=USE_BBOX,
        bbox_padding=BBOX_PADDING,
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        **dataloader_kwargs(num_workers),
    )


seed_everything(RANDOM_SEED)


In [ ]:
class_to_idx = load_json(CLASS_TO_IDX_PATH)
idx_to_class = {idx: name for name, idx in class_to_idx.items()}

train_records = load_json(TRAIN_RECORDS_PATH)
val_records = load_json(VAL_RECORDS_PATH)
test_records = load_json(TEST_RECORDS_PATH)

num_classes = len(class_to_idx)
breed_names = [
    idx_to_class[i].split("-", 1)[1].replace("_", " ")
    for i in range(num_classes)
]

split_records = {
    "train": train_records,
    "val": val_records,
    "test": test_records,
}

eval_transform = transforms.Compose(
    [
        transforms.Resize(RESIZE_SIZE, interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

loaders = {
    split: make_loader(records, eval_transform, shuffle=False, num_workers=NUM_WORKERS)
    for split, records in split_records.items()
}

print(f"Num classes   : {num_classes}")
for split, records in split_records.items():
    print(f"{split:5s} samples : {len(records):5d} | batches: {len(loaders[split]):4d}")


## 3. Kiểm thử dữ liệu

Các kiểm thử này không cần deep learning. Chúng bắt những lỗi nền tảng nhất: thiếu file, label sai range, split bị trùng ảnh, class không phủ đủ trong từng split, tensor sau transform bị NaN/Inf.


In [ ]:
def add_test_result(results: list[dict], name: str, passed: bool, detail: str) -> None:
    results.append({"test": name, "passed": bool(passed), "detail": detail})


def count_missing_paths(records: list[dict], key: str) -> int:
    return sum(1 for record in records if not Path(record[key]).exists())


def run_data_integrity_tests() -> list[dict]:
    results: list[dict] = []

    required_files = [
        CLASS_TO_IDX_PATH,
        TRAIN_RECORDS_PATH,
        VAL_RECORDS_PATH,
        TEST_RECORDS_PATH,
    ]
    missing = [str(path) for path in required_files if not path.exists()]
    add_test_result(
        results,
        "required_artifact_files",
        len(missing) == 0,
        "OK" if not missing else f"Missing: {missing}",
    )

    add_test_result(
        results,
        "num_classes",
        num_classes == 120,
        f"num_classes={num_classes}",
    )

    for split, records in split_records.items():
        labels = [record["class_id"] for record in records]
        label_min, label_max = min(labels), max(labels)
        label_ok = label_min >= 0 and label_max < num_classes
        add_test_result(
            results,
            f"{split}_label_range",
            label_ok,
            f"min={label_min}, max={label_max}",
        )

        class_counts = Counter(labels)
        missing_classes = sorted(set(range(num_classes)) - set(class_counts))
        add_test_result(
            results,
            f"{split}_class_coverage",
            len(missing_classes) == 0,
            "OK" if not missing_classes else f"missing_classes={missing_classes[:10]}",
        )

        missing_images = count_missing_paths(records, "image_path")
        add_test_result(
            results,
            f"{split}_image_paths_exist",
            missing_images == 0,
            f"missing_images={missing_images}",
        )

        missing_annotations = count_missing_paths(records, "annotation_path")
        add_test_result(
            results,
            f"{split}_annotation_paths_exist",
            missing_annotations == 0,
            f"missing_annotations={missing_annotations}",
        )

        counts = np.array(list(class_counts.values()))
        add_test_result(
            results,
            f"{split}_class_balance_summary",
            True,
            f"min={counts.min()}, median={np.median(counts):.1f}, max={counts.max()}",
        )

    image_sets = {
        split: set(record["image_path"] for record in records)
        for split, records in split_records.items()
    }
    leakage_pairs = []
    for left, right in [("train", "val"), ("train", "test"), ("val", "test")]:
        overlap = image_sets[left] & image_sets[right]
        if overlap:
            leakage_pairs.append((left, right, len(overlap)))
    add_test_result(
        results,
        "split_leakage_by_image_path",
        len(leakage_pairs) == 0,
        "OK" if not leakage_pairs else str(leakage_pairs),
    )

    images, labels = next(iter(loaders["test"]))
    tensor_ok = images.shape[1:] == (3, IMAGE_SIZE, IMAGE_SIZE) and torch.isfinite(images).all().item()
    label_ok = labels.ndim == 1 and labels.min().item() >= 0 and labels.max().item() < num_classes
    add_test_result(
        results,
        "eval_batch_shape_and_finite_values",
        tensor_ok and label_ok,
        f"images={tuple(images.shape)}, labels={tuple(labels.shape)}",
    )

    return results


data_test_results = run_data_integrity_tests()

print(f"{'Test':38s} | {'Status':6s} | Detail")
print("-" * 95)
for item in data_test_results:
    status = "PASS" if item["passed"] else "FAIL"
    print(f"{item['test']:38s} | {status:6s} | {item['detail']}")

assert all(item["passed"] for item in data_test_results if not item["test"].endswith("summary")), "Có data test không pass. Cần xử lý trước khi đánh giá model."


## 4. Model registry và kiểm thử checkpoint

Registry phải khớp kiến trúc đã train ở các tuần trước. Notebook sẽ skip checkpoint chưa tồn tại để vẫn chạy được khi mới chỉ train AlexNet.


In [ ]:
class AlexNetFromScratch(nn.Module):
    def __init__(self, num_classes: int, dropout: float = 0.5) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(64, 192, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        self.avgpool = nn.AdaptiveAvgPool2d((6, 6))
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


def build_transfer_model(backbone_name: str, num_classes: int) -> nn.Module:
    dropout = 0.3
    if backbone_name == "resnet50":
        model = models.resnet50(weights=None)
        model.fc = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(2048, num_classes))
    elif backbone_name == "mobilenet_v2":
        model = models.mobilenet_v2(weights=None)
        model.classifier = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(1280, num_classes))
    elif backbone_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=None)
        model.classifier = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(1280, num_classes))
    else:
        raise ValueError(f"Unknown backbone: {backbone_name}")
    return model


MODEL_REGISTRY: dict[str, dict] = {
    "alexnet_from_scratch": {
        "display_name": "AlexNet (scratch)",
        "builder": lambda nc: AlexNetFromScratch(nc, dropout=0.5),
        "checkpoint": CHECKPOINT_DIR / "alexnet_from_scratch_best.pt",
        "history": HISTORY_DIR / "alexnet_from_scratch_history.json",
        "source_week": "week67",
    },
    "resnet50_transfer": {
        "display_name": "ResNet50 (transfer)",
        "builder": lambda nc: build_transfer_model("resnet50", nc),
        "checkpoint": CHECKPOINT_DIR / "resnet50_best.pt",
        "history": HISTORY_DIR / "resnet50_history.json",
        "source_week": "week89",
    },
    "mobilenet_v2_transfer": {
        "display_name": "MobileNetV2 (transfer)",
        "builder": lambda nc: build_transfer_model("mobilenet_v2", nc),
        "checkpoint": CHECKPOINT_DIR / "mobilenet_v2_best.pt",
        "history": HISTORY_DIR / "mobilenet_v2_history.json",
        "source_week": "week89",
    },
    "efficientnet_b0_transfer": {
        "display_name": "EfficientNet-B0 (transfer)",
        "builder": lambda nc: build_transfer_model("efficientnet_b0", nc),
        "checkpoint": CHECKPOINT_DIR / "efficientnet_b0_best.pt",
        "history": HISTORY_DIR / "efficientnet_b0_history.json",
        "source_week": "week89",
    },
}


def safe_torch_load(path: Path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def load_model_from_registry(model_key: str, device: torch.device) -> tuple[nn.Module, dict]:
    cfg = MODEL_REGISTRY[model_key]
    checkpoint = safe_torch_load(cfg["checkpoint"], map_location=device)
    model = cfg["builder"](num_classes)
    model.load_state_dict(checkpoint["model_state_dict"])
    model = model.to(device)
    model.eval()
    return model, checkpoint


available_model_keys = []
print(f"{'Model':28s} | {'Checkpoint':34s} | Status")
print("-" * 82)
for model_key, cfg in MODEL_REGISTRY.items():
    exists = cfg["checkpoint"].exists()
    if exists:
        available_model_keys.append(model_key)
    status = "FOUND" if exists else "SKIP"
    print(f"{cfg['display_name']:28s} | {cfg['checkpoint'].name:34s} | {status}")

print(f"\nAvailable models: {len(available_model_keys)}")


In [ ]:
def run_checkpoint_smoke_tests() -> list[dict]:
    results: list[dict] = []
    sample_images, _ = next(iter(loaders["test"]))
    sample_images = sample_images[:2].to(DEVICE, non_blocking=True)

    for model_key in available_model_keys:
        cfg = MODEL_REGISTRY[model_key]
        try:
            model, checkpoint = load_model_from_registry(model_key, DEVICE)
            with torch.inference_mode():
                logits = model(sample_images)
            output_ok = logits.shape == (sample_images.shape[0], num_classes)
            finite_ok = torch.isfinite(logits).all().item()
            add_test_result(
                results,
                f"{model_key}_checkpoint_forward",
                output_ok and finite_ok,
                f"shape={tuple(logits.shape)}, params={count_parameters(model):,}, epoch={checkpoint.get('epoch', '?')}",
            )
            del model, checkpoint, logits
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception as exc:
            add_test_result(results, f"{model_key}_checkpoint_forward", False, repr(exc))

    return results


checkpoint_test_results = run_checkpoint_smoke_tests()
print(f"{'Test':42s} | {'Status':6s} | Detail")
print("-" * 105)
for item in checkpoint_test_results:
    status = "PASS" if item["passed"] else "FAIL"
    print(f"{item['test']:42s} | {status:6s} | {item['detail']}")

assert all(item["passed"] for item in checkpoint_test_results), "Có checkpoint/model smoke test không pass."


## 5. Benchmark DataLoader và inference

Tối ưu hệ thống ở đây chỉ xử lý phần có tác động rõ ràng trong project này: DataLoader workers, pinned memory, persistent workers, throughput inference và peak GPU memory. Nếu model còn yếu về accuracy, tối ưu hệ thống không thay thế được cải thiện dữ liệu/model.


In [ ]:
def benchmark_dataloader_workers(
    records: list[dict],
    transform,
    worker_candidates: list[int],
    max_batches: int = 60,
) -> list[dict]:
    results = []
    for num_workers in worker_candidates:
        if os.cpu_count() is not None and num_workers > os.cpu_count():
            continue
        loader = make_loader(
            records,
            transform,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=num_workers,
        )
        t0 = time.perf_counter()
        batches = 0
        samples = 0
        for images, labels in loader:
            batches += 1
            samples += labels.shape[0]
            if batches >= max_batches:
                break
        elapsed = time.perf_counter() - t0
        result = {
            "num_workers": num_workers,
            "batches": batches,
            "samples": samples,
            "seconds": elapsed,
            "images_per_sec": samples / elapsed if elapsed > 0 else float("nan"),
            "batches_per_sec": batches / elapsed if elapsed > 0 else float("nan"),
        }
        results.append(result)
        del loader
    return results


loader_benchmark = benchmark_dataloader_workers(
    train_records,
    eval_transform,
    BENCHMARK_WORKERS,
    max_batches=BENCHMARK_BATCHES,
)

print(f"{'workers':>7s} | {'seconds':>8s} | {'img/s':>10s} | {'batch/s':>8s}")
print("-" * 45)
for row in loader_benchmark:
    print(
        f"{row['num_workers']:7d} | {row['seconds']:8.2f} | "
        f"{row['images_per_sec']:10.1f} | {row['batches_per_sec']:8.2f}"
    )

if loader_benchmark:
    best = max(loader_benchmark, key=lambda row: row["images_per_sec"])
    print(f"\nBest observed workers: {best['num_workers']} ({best['images_per_sec']:.1f} img/s)")
    if best["num_workers"] != NUM_WORKERS:
        print(f"Current NUM_WORKERS={NUM_WORKERS}; cân nhắc cập nhật nếu benchmark này ổn định qua nhiều lần chạy.")


In [ ]:
@torch.inference_mode()
def benchmark_inference(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    max_batches: int = 40,
    use_amp: bool = False,
) -> dict:
    model.eval()
    warmup_batches = 2 if device.type == "cuda" else 0

    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    total_samples = 0
    timed_batches = 0
    started = None

    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        if device.type == "cuda":
            torch.cuda.synchronize()

        if batch_idx == warmup_batches:
            started = time.perf_counter()

        with torch.amp.autocast(device_type=device.type, enabled=use_amp and device.type == "cuda"):
            _ = model(images)

        if batch_idx >= warmup_batches:
            timed_batches += 1
            total_samples += images.shape[0]

        if timed_batches >= max_batches:
            break

    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - started if started is not None else float("nan")
    peak_memory_mib = None
    if device.type == "cuda":
        peak_memory_mib = torch.cuda.max_memory_reserved() / 1024**2

    return {
        "batches": timed_batches,
        "samples": total_samples,
        "seconds": elapsed,
        "images_per_sec": total_samples / elapsed if elapsed > 0 else float("nan"),
        "ms_per_batch": 1000 * elapsed / timed_batches if timed_batches > 0 else float("nan"),
        "peak_memory_mib": peak_memory_mib,
    }


inference_benchmarks = {}
for model_key in available_model_keys:
    cfg = MODEL_REGISTRY[model_key]
    model, checkpoint = load_model_from_registry(model_key, DEVICE)
    result = benchmark_inference(
        model,
        loaders["test"],
        DEVICE,
        max_batches=INFERENCE_BENCHMARK_BATCHES,
        use_amp=EVAL_USE_AMP,
    )
    inference_benchmarks[model_key] = result
    mem_text = "N/A" if result["peak_memory_mib"] is None else f"{result['peak_memory_mib']:.1f} MiB"
    print(
        f"{cfg['display_name']:28s} | "
        f"{result['images_per_sec']:8.1f} img/s | "
        f"{result['ms_per_batch']:7.2f} ms/batch | peak_reserved={mem_text}"
    )
    del model, checkpoint
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 6. Đánh giá thực nghiệm trên val/test

Metric chính gồm loss, Top-1, Top-5, precision/recall/F1 macro và weighted. Macro metric quan trọng trong fine-grained classification vì mỗi class cần được xem tương đối công bằng, không chỉ theo số lượng mẫu.


In [ ]:
@torch.inference_mode()
def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    max_batches: int | None = None,
    use_amp: bool = False,
) -> dict:
    model.eval()
    criterion = nn.CrossEntropyLoss()

    all_true = []
    all_pred = []
    all_conf = []
    loss_sum = 0.0
    top1_count = 0
    top5_count = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels_device = labels.to(device, non_blocking=True)

        with torch.amp.autocast(device_type=device.type, enabled=use_amp and device.type == "cuda"):
            logits = model(images)
            loss = criterion(logits, labels_device)

        probs = torch.softmax(logits.float(), dim=1)
        conf, preds = probs.max(dim=1)
        top5 = logits.topk(k=min(5, logits.shape[1]), dim=1).indices

        batch_size = labels.shape[0]
        loss_sum += loss.item() * batch_size
        top1_count += (preds.cpu() == labels).sum().item()
        top5_count += top5.cpu().eq(labels.view(-1, 1)).any(dim=1).sum().item()
        total += batch_size

        all_true.append(labels.numpy())
        all_pred.append(preds.cpu().numpy())
        all_conf.append(conf.cpu().numpy())

        if max_batches is not None and batch_idx + 1 >= max_batches:
            break

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)
    confidence = np.concatenate(all_conf)

    return {
        "loss": loss_sum / total,
        "top1": top1_count / total,
        "top5": top5_count / total,
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "mean_confidence": float(confidence.mean()),
        "y_true": y_true,
        "y_pred": y_pred,
        "confidence": confidence,
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=list(range(num_classes))),
    }


def serializable_metrics(metrics: dict) -> dict:
    keys = [
        "loss", "top1", "top5", "precision_macro", "precision_weighted",
        "recall_macro", "recall_weighted", "f1_macro", "f1_weighted", "mean_confidence",
    ]
    return {key: float(metrics[key]) for key in keys}


experiment_results: dict[str, dict] = {}
for model_key in available_model_keys:
    cfg = MODEL_REGISTRY[model_key]
    print(f"\n{'=' * 78}")
    print(cfg["display_name"])
    print(f"{'=' * 78}")

    model, checkpoint = load_model_from_registry(model_key, DEVICE)
    model_results = {
        "display_name": cfg["display_name"],
        "source_week": cfg["source_week"],
        "params": count_parameters(model),
        "checkpoint": str(cfg["checkpoint"]),
        "best_epoch": checkpoint.get("epoch"),
        "best_val_top1_in_checkpoint": checkpoint.get("best_val_top1"),
    }

    for split in ["val", "test"]:
        metrics = evaluate_model(model, loaders[split], DEVICE, use_amp=EVAL_USE_AMP)
        model_results[split] = metrics
        print(
            f"{split.upper():4s} | loss={metrics['loss']:.4f} | "
            f"top1={metrics['top1']:.4f} | top5={metrics['top5']:.4f} | "
            f"f1_macro={metrics['f1_macro']:.4f} | conf={metrics['mean_confidence']:.4f}"
        )

    experiment_results[model_key] = model_results
    del model, checkpoint
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\nEvaluated models: {len(experiment_results)}")


In [ ]:
if not experiment_results:
    print("Không có model nào để tổng hợp. Hãy train ít nhất một checkpoint trước.")
else:
    header = (
        f"{'Model':28s} | {'Split':4s} | {'Loss':>7s} | {'Top1':>7s} | {'Top5':>7s} | "
        f"{'F1 macro':>8s} | {'F1 wt':>7s} | {'Conf':>6s}"
    )
    print(header)
    print("-" * len(header))
    for model_key, result in experiment_results.items():
        for split in ["val", "test"]:
            m = result[split]
            print(
                f"{result['display_name']:28s} | {split:4s} | "
                f"{m['loss']:7.4f} | {m['top1']:7.4f} | {m['top5']:7.4f} | "
                f"{m['f1_macro']:8.4f} | {m['f1_weighted']:7.4f} | {m['mean_confidence']:6.4f}"
            )
        print("-" * len(header))


In [ ]:
if experiment_results:
    model_names = [result["display_name"] for result in experiment_results.values()]
    x = np.arange(len(model_names))
    width = 0.35

    fig, axes = plt.subplots(1, 2, figsize=(max(10, len(model_names) * 3), 5))

    for ax, metric_key, title, ylabel in [
        (axes[0], "top1", "Top-1 Accuracy", "Accuracy (%)"),
        (axes[1], "f1_macro", "Macro F1", "F1 (%)"),
    ]:
        val_scores = [result["val"][metric_key] * 100 for result in experiment_results.values()]
        test_scores = [result["test"][metric_key] * 100 for result in experiment_results.values()]
        ax.bar(x - width / 2, val_scores, width, label="val", color="#4c72b0")
        ax.bar(x + width / 2, test_scores, width, label="test", color="#55a868")
        ax.set_xticks(x)
        ax.set_xticklabels(model_names, rotation=10, ha="right")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.set_ylim(0, 105)
        ax.grid(axis="y", alpha=0.25)
        ax.legend()

    plt.tight_layout()
    plt.show()


## 7. Kiểm tra độ ổn định khi chạy lặp lại

Một model ở `eval()` mode không nên đổi dự đoán khi chạy lại cùng input. Phần này kiểm tra hai mức:

- Cùng một batch chạy nhiều lần: so sánh prediction agreement và sai khác logit lớn nhất.
- Cùng một subset đánh giá chạy nhiều lần: metric Top-1/Top-5/loss có dao động không.


In [ ]:
@torch.inference_mode()
def repeated_batch_stability(
    model: nn.Module,
    batch: torch.Tensor,
    repeats: int = 5,
    use_amp: bool = False,
) -> dict:
    model.eval()
    batch = batch.to(DEVICE, non_blocking=True)

    logits_runs = []
    pred_runs = []
    for _ in range(repeats):
        with torch.amp.autocast(device_type=DEVICE.type, enabled=use_amp and DEVICE.type == "cuda"):
            logits = model(batch)
        logits = logits.float().cpu()
        logits_runs.append(logits)
        pred_runs.append(logits.argmax(dim=1))

    reference_logits = logits_runs[0]
    reference_preds = pred_runs[0]
    max_abs_logit_delta = max(
        (run - reference_logits).abs().max().item()
        for run in logits_runs[1:]
    ) if repeats > 1 else 0.0
    prediction_agreements = [
        (pred == reference_preds).float().mean().item()
        for pred in pred_runs[1:]
    ]

    return {
        "repeats": repeats,
        "prediction_agreement_mean": float(np.mean(prediction_agreements)) if prediction_agreements else 1.0,
        "prediction_agreement_min": float(np.min(prediction_agreements)) if prediction_agreements else 1.0,
        "max_abs_logit_delta": float(max_abs_logit_delta),
    }


def repeated_metric_stability(
    model: nn.Module,
    loader: DataLoader,
    repeats: int = 3,
    max_batches: int = 8,
) -> dict:
    runs = []
    for repeat in range(repeats):
        seed_everything(RANDOM_SEED + repeat)
        metrics = evaluate_model(
            model,
            loader,
            DEVICE,
            max_batches=max_batches,
            use_amp=EVAL_USE_AMP,
        )
        runs.append(metrics)

    return {
        "repeats": repeats,
        "max_batches": max_batches,
        "loss_values": [float(run["loss"]) for run in runs],
        "top1_values": [float(run["top1"]) for run in runs],
        "top5_values": [float(run["top5"]) for run in runs],
        "loss_std": float(np.std([run["loss"] for run in runs])),
        "top1_std": float(np.std([run["top1"] for run in runs])),
        "top5_std": float(np.std([run["top5"] for run in runs])),
    }


stability_results = {}
sample_batch, _ = next(iter(loaders["test"]))
sample_batch = sample_batch[: min(16, sample_batch.shape[0])]

for model_key in available_model_keys:
    cfg = MODEL_REGISTRY[model_key]
    model, checkpoint = load_model_from_registry(model_key, DEVICE)
    batch_stability = repeated_batch_stability(
        model,
        sample_batch,
        repeats=5,
        use_amp=EVAL_USE_AMP,
    )
    metric_stability = repeated_metric_stability(
        model,
        loaders["test"],
        repeats=STABILITY_REPEATS,
        max_batches=STABILITY_MAX_BATCHES,
    )
    stability_results[model_key] = {
        "batch": batch_stability,
        "metric": metric_stability,
    }

    print(f"\n{cfg['display_name']}")
    print(
        f"  Batch repeat: agreement_mean={batch_stability['prediction_agreement_mean']:.4f}, "
        f"agreement_min={batch_stability['prediction_agreement_min']:.4f}, "
        f"max_abs_logit_delta={batch_stability['max_abs_logit_delta']:.8f}"
    )
    print(
        f"  Metric repeat: top1_std={metric_stability['top1_std']:.8f}, "
        f"top5_std={metric_stability['top5_std']:.8f}, loss_std={metric_stability['loss_std']:.8f}"
    )

    del model, checkpoint
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 8. Robustness nhẹ với test-time augmentation

Metric chính dùng crop cố định. Cell này trả lời câu hỏi khác: nếu ảnh bị crop/flip nhẹ, dự đoán có giữ nguyên không? Đây không phải accuracy chính thức, mà là một kiểm thử ổn định hành vi.


In [ ]:
tta_transform = transforms.Compose(
    [
        transforms.Resize(RESIZE_SIZE, interpolation=InterpolationMode.BICUBIC),
        transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.90, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)


@torch.inference_mode()
def collect_predictions_for_records(
    model: nn.Module,
    records: list[dict],
    transform,
    batch_size: int = BATCH_SIZE,
    seed: int | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    if seed is not None:
        seed_everything(seed)
    loader = make_loader(
        records,
        transform,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )
    preds = []
    confs = []
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        logits = model(images)
        probs = torch.softmax(logits.float(), dim=1)
        conf, pred = probs.max(dim=1)
        preds.append(pred.cpu().numpy())
        confs.append(conf.cpu().numpy())
    return np.concatenate(preds), np.concatenate(confs)


tta_records = test_records[: min(TTA_MAX_IMAGES, len(test_records))]
tta_results = {}

for model_key in available_model_keys:
    cfg = MODEL_REGISTRY[model_key]
    model, checkpoint = load_model_from_registry(model_key, DEVICE)

    base_preds, base_confs = collect_predictions_for_records(
        model,
        tta_records,
        eval_transform,
        seed=RANDOM_SEED,
    )

    agreements = []
    confidence_means = []
    for sample_idx in range(TTA_SAMPLES):
        aug_preds, aug_confs = collect_predictions_for_records(
            model,
            tta_records,
            tta_transform,
            seed=RANDOM_SEED + 100 + sample_idx,
        )
        agreements.append(float((aug_preds == base_preds).mean()))
        confidence_means.append(float(aug_confs.mean()))

    result = {
        "num_images": len(tta_records),
        "tta_samples": TTA_SAMPLES,
        "agreement_mean": float(np.mean(agreements)),
        "agreement_min": float(np.min(agreements)),
        "agreement_values": agreements,
        "base_confidence_mean": float(base_confs.mean()),
        "tta_confidence_mean": float(np.mean(confidence_means)),
        "tta_confidence_std": float(np.std(confidence_means)),
    }
    tta_results[model_key] = result

    print(
        f"{cfg['display_name']:28s} | "
        f"agreement_mean={result['agreement_mean']:.4f} | "
        f"agreement_min={result['agreement_min']:.4f} | "
        f"base_conf={result['base_confidence_mean']:.4f} | "
        f"tta_conf={result['tta_confidence_mean']:.4f}"
    )

    del model, checkpoint
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 9. Phân tích lỗi: class khó và cặp nhầm lẫn

Với fine-grained classification, nhìn accuracy tổng là chưa đủ. Cần xem class nào thường sai và sai sang class nào. Đây là phần giúp biến metric thành nhận định thực nghiệm.


In [ ]:
def per_class_accuracy_from_cm(cm: np.ndarray) -> np.ndarray:
    totals = cm.sum(axis=1)
    correct = np.diag(cm)
    return np.divide(correct, totals, out=np.zeros_like(correct, dtype=float), where=totals > 0)


def top_confused_pairs(cm: np.ndarray, class_names: list[str], top_k: int = 15) -> list[tuple[int, str, str]]:
    matrix = cm.copy()
    np.fill_diagonal(matrix, 0)
    pairs = []
    for true_idx in range(matrix.shape[0]):
        for pred_idx in range(matrix.shape[1]):
            count = int(matrix[true_idx, pred_idx])
            if count > 0:
                pairs.append((count, class_names[true_idx], class_names[pred_idx]))
    return sorted(pairs, reverse=True)[:top_k]


if experiment_results:
    best_model_key = max(experiment_results, key=lambda key: experiment_results[key]["test"]["top1"])
    best_result = experiment_results[best_model_key]
    best_display_name = best_result["display_name"]
    cm = best_result["test"]["confusion_matrix"]
    per_class_acc = per_class_accuracy_from_cm(cm)
    sorted_indices = np.argsort(per_class_acc)

    print(f"Best model by test Top-1: {best_display_name}")
    print("\nHardest classes by per-class accuracy:")
    for rank, idx in enumerate(sorted_indices[:10], 1):
        print(f"  {rank:2d}. {breed_names[idx]:30s} acc={per_class_acc[idx]:.4f}")

    print("\nEasiest classes by per-class accuracy:")
    for rank, idx in enumerate(sorted_indices[-10:][::-1], 1):
        print(f"  {rank:2d}. {breed_names[idx]:30s} acc={per_class_acc[idx]:.4f}")

    print("\nTop confused pairs:")
    for count, true_name, pred_name in top_confused_pairs(cm, breed_names, top_k=15):
        print(f"  {count:3d} | true={true_name:28s} -> pred={pred_name}")


In [ ]:
if experiment_results:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    hardest = sorted_indices[:10]
    easiest = sorted_indices[-10:][::-1]

    axes[0].barh(range(len(hardest)), per_class_acc[hardest] * 100, color="#c44e52")
    axes[0].set_yticks(range(len(hardest)))
    axes[0].set_yticklabels([breed_names[i] for i in hardest], fontsize=9)
    axes[0].set_xlabel("Accuracy (%)")
    axes[0].set_title(f"Hardest classes - {best_display_name}")
    axes[0].set_xlim(0, 105)
    axes[0].invert_yaxis()
    axes[0].grid(axis="x", alpha=0.25)

    axes[1].barh(range(len(easiest)), per_class_acc[easiest] * 100, color="#55a868")
    axes[1].set_yticks(range(len(easiest)))
    axes[1].set_yticklabels([breed_names[i] for i in easiest], fontsize=9)
    axes[1].set_xlabel("Accuracy (%)")
    axes[1].set_title(f"Easiest classes - {best_display_name}")
    axes[1].set_xlim(0, 105)
    axes[1].invert_yaxis()
    axes[1].grid(axis="x", alpha=0.25)

    plt.tight_layout()
    plt.show()


In [ ]:
def show_misclassified_examples(
    records: list[dict],
    y_true: np.ndarray,
    y_pred: np.ndarray,
    confidence: np.ndarray,
    class_names: list[str],
    max_examples: int = 12,
) -> None:
    wrong_indices = np.where(y_true != y_pred)[0]
    if len(wrong_indices) == 0:
        print("Không có mẫu sai trong split này.")
        return

    # Ưu tiên lỗi có confidence cao vì đây thường là lỗi đáng phân tích nhất.
    ranked = sorted(wrong_indices, key=lambda idx: confidence[idx], reverse=True)[:max_examples]
    cols = 4
    rows = int(np.ceil(len(ranked) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4.2 * cols, 4.2 * rows))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for ax, idx in zip(axes, ranked):
        record = records[idx]
        image = Image.open(record["image_path"]).convert("RGB")
        ax.imshow(image)
        true_name = class_names[int(y_true[idx])]
        pred_name = class_names[int(y_pred[idx])]
        ax.set_title(
            f"True: {true_name}\nPred: {pred_name}\nconf={confidence[idx]:.2f}",
            fontsize=9,
            color="#9b2226",
        )

    plt.tight_layout()
    plt.show()


if experiment_results:
    show_misclassified_examples(
        test_records,
        best_result["test"]["y_true"],
        best_result["test"]["y_pred"],
        best_result["test"]["confidence"],
        breed_names,
        max_examples=ERROR_EXAMPLE_COUNT,
    )


## 10. Classification report cho model tốt nhất

Report này dài vì có 120 class. Khi viết báo cáo, không cần paste toàn bộ nếu quá dài; nên trích summary và các class có vấn đề nhất.


In [ ]:
if experiment_results:
    print(f"Classification report - {best_display_name} (test set)\n")
    print(
        classification_report(
            best_result["test"]["y_true"],
            best_result["test"]["y_pred"],
            target_names=breed_names,
            digits=4,
            zero_division=0,
        )
    )


## 11. Xuất kết quả kiểm thử

File JSON giúp dùng lại kết quả trong báo cáo hoặc so sánh giữa các lần chạy. Các mảng lớn như confusion matrix và prediction thô không được export toàn bộ để file nhỏ và dễ đọc.


In [ ]:
def make_json_safe(value):
    if isinstance(value, dict):
        return {str(k): make_json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [make_json_safe(v) for v in value]
    if isinstance(value, tuple):
        return [make_json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    return value


export_payload = {
    "config": {
        "image_size": IMAGE_SIZE,
        "resize_size": RESIZE_SIZE,
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "device": str(DEVICE),
        "eval_use_amp": EVAL_USE_AMP,
    },
    "data_tests": data_test_results,
    "checkpoint_tests": checkpoint_test_results,
    "dataloader_benchmark": loader_benchmark,
    "inference_benchmarks": inference_benchmarks,
    "stability": stability_results,
    "tta_robustness": tta_results,
    "models": {},
}

for model_key, result in experiment_results.items():
    export_payload["models"][model_key] = {
        "display_name": result["display_name"],
        "source_week": result["source_week"],
        "params": result["params"],
        "checkpoint": result["checkpoint"],
        "best_epoch": result["best_epoch"],
        "best_val_top1_in_checkpoint": result["best_val_top1_in_checkpoint"],
        "val": serializable_metrics(result["val"]),
        "test": serializable_metrics(result["test"]),
    }

output_path = HISTORY_DIR / "week11_system_test_results.json"
output_path.write_text(
    json.dumps(make_json_safe(export_payload), indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(f"Saved Week 11 test results to: {output_path}")


## 12. Kết luận sau khi chạy

Khi viết báo cáo Week 11, nên trả lời ngắn gọn các câu hỏi sau:

1. Data tests có pass hết không? Nếu không, lỗi nào ảnh hưởng trực tiếp đến metric?
2. Checkpoint nào load được, checkpoint nào chưa có?
3. `NUM_WORKERS` hiện tại có hợp lý theo benchmark không?
4. Test Top-1/Top-5/F1 macro của model tốt nhất là bao nhiêu?
5. Chạy lặp lại có ổn định không: prediction agreement, metric std?
6. Class nào khó nhất và các cặp nhầm lẫn phổ biến là gì?
7. Nếu model chưa tốt, nguyên nhân nhiều khả năng nằm ở dữ liệu/model/training hay ở hệ thống inference?

Với project foundation, kết luận tốt không chỉ là số accuracy. Quan trọng là chứng minh pipeline đánh giá có thể tin được và chỉ ra bước cải thiện tiếp theo dựa trên bằng chứng.
